# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srinivas25046/FlyRank-MLstarter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / Scoring**

My lane is **Refresh / Content Opportunity Scoring**, and the underlying decision from ML-02 is *"which pages should be reviewed first?"* — not *"is this page declining, yes or no?"*.
The `framing-ml-problems` table maps "which ones first?" straight to **ranking/scoring**, with a priority score as the target, not a class label.

Classification is tempting because `trend_direction == "down"` already looks like a ready-made label. But a binary label throws away magnitude: a page that dropped 90% while pulling 50,000 impressions and a page that dropped 22% on 20 impressions both get labeled `1`, even though no reviewer would treat them the same. The manager needs an ordered queue, not a yes/no verdict — that's what makes this scoring/ranking rather than classification.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

RAW_URL = (
    "https://raw.githubusercontent.com/Srinivas25046/FlyRank-MLstarter/"
    "refs/heads/main/data/raw/content_refresh_anonymized.csv"
)
LOCAL_PATH = "../../data/raw/content_refresh_anonymized.csv"

try:
    # Works when running from a real clone of the repo (e.g. work/notebooks/ locally)
    df = pd.read_csv(LOCAL_PATH)
except FileNotFoundError:
    # Works in Colab, or anywhere else the repo isn't checked out on disk
    df = pd.read_csv(RAW_URL)

# Show why a binary label alone can't support "which ones first?"
declining = df[df["trend_direction"] == "down"]
print(f"Pages labeled 'down': {len(declining):,} ({len(declining)/len(df)*100:.1f}% of all pages)")
print("\nAll of these get the SAME binary label, but look at the spread in traffic at stake:")
print(declining["impressions_90d"].describe()[["min", "25%", "50%", "75%", "max"]])
print(
    "\n-> One classification bucket spans from 1 impression to "
    f"{int(declining['impressions_90d'].max()):,} impressions in 90 days. "
    "A single label can't order these — a score can."
)

Pages labeled 'down': 16,262 (54.2% of all pages)

All of these get the SAME binary label, but look at the spread in traffic at stake:
min         1.00
25%       179.00
50%       961.00
75%      3831.75
max    517715.00
Name: impressions_90d, dtype: float64

-> One classification bucket spans from 1 impression to 517,715 impressions in 90 days. A single label can't order these — a score can.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `priority_score` (a continuous proxy, not an observed outcome)**

No column in this dataset records "a human reviewed this page and it was worth it" — that
outcome has never been observed here, so I can't pretend to have a real target yet. What I can
build is an honest **proxy**, following the two framing rules (name the metric, and don't let a
defined rule pretend to be the world):

- **Decline signal** — same definition the data dictionary gives for `is_declining_label`
  (`trend_direction == "down"`). The raw CSV doesn't ship that derived column, so I rebuild it
  here from `trend_direction` directly, in the open.
- **Traffic at stake** — `impressions_90d` and `clicks_90d`, because a declining page nobody
  searches for isn't worth a reviewer's hour.
- **Measurability gate** — mirrors `measurable_opportunity` in the dictionary
  (`impressions_90d >= 100` and `sessions_90d > 0`), so tiny-sample noise doesn't rank a page
  highly by accident.

`priority_score = is_declining * log1p(impressions_90d + clicks_90d) * measurable_opportunity`

This is a **defined proxy**, not an observed one — I'm saying so explicitly rather than dressing
it up. The honest path to an observed target: once the FlyRank warehouse's
`fact_content_daily_performance` table is in play (week 4+), I could check whether pages that
were actually reviewed/refreshed saw impressions recover in the *following* 30–90 day window,
using only `*_prev30`-style columns to avoid leakage. That's next week's job, not this one's.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Build the proxy target from observed, documented rules — no ML yet, just the label itself.
df["is_declining"] = (df["trend_direction"] == "down").astype(int)
df["measurable_opportunity"] = (
    (df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)
).astype(int)

import numpy as np
df["priority_score"] = (
    df["is_declining"]
    * np.log1p(df["impressions_90d"] + df["clicks_90d"])
    * df["measurable_opportunity"]
)

print(f"Pages flagged declining:            {df['is_declining'].sum():,}")
print(f"Pages passing the measurability gate: {df['measurable_opportunity'].sum():,}")
print(f"Pages with priority_score > 0:        {(df['priority_score'] > 0).sum():,}")
print("\nTop 5 pages by priority_score:")
print(
    df.sort_values("priority_score", ascending=False)
    [["content_id", "trend_direction", "impressions_90d", "clicks_90d", "priority_score"]]
    .head(5)
    .to_string(index=False)
)

Pages flagged declining:            16,262
Pages passing the measurability gate: 22,006
Pages with priority_score > 0:        13,152

Top 5 pages by priority_score:
          content_id trend_direction  impressions_90d  clicks_90d  priority_score
content_5fe46e04994d            down           517715         741       13.158612
content_8c19996aa890            down           509252         785       13.142241
content_4c36c775b818            down           463103        1889       13.049778
content_1a9e894be2e2            down           416180         944       12.941141
content_2c2606c5d176            down           347399        1854       12.763555


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@K, K = top 100 pages (~top 0.3% of the queue)**

Because the output is a ranked queue and not a class prediction, accuracy or plain AUC would be
the wrong number — they score every page equally, but a reviewer only ever looks at the top of
the list. Precision@K answers the question a manager actually asks: *"of the 100 pages you put
in front of me today, how many were real, worth-my-time opportunities?"* That's a number I can
defend in a sentence — "the top 100 recommendations were right 91% of the time" — in a way a
global AUC score never is.

Today, with only the starter slice, "real opportunity" has to mean the same proxy definition
from Section 2 (declining + measurable) — which makes Precision@K against that same definition
trivially high right now, since the queue is scored directly from the thing it's being checked
against. I'm not hiding that circularity: it's exactly why this number isn't the real test yet.
What *is* meaningful today is comparing the proxy queue to a rule with zero traffic-awareness
(oldest-content-first) on that same yardstick — a real gap there means the extra signals are
doing something. Once forward-looking warehouse data exists, the honest upgrade is to redefine
"real opportunity" as *pages that actually recovered impressions in the next 30–90 days* — an
outcome independent of how the queue was built — and Precision@K becomes the real test.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

K = 100

top_k = df.sort_values("priority_score", ascending=False).head(K)
is_real_opportunity = (top_k["is_declining"] == 1) & (top_k["measurable_opportunity"] == 1)

precision_at_k = is_real_opportunity.mean()
print(f"Precision@{K} of the priority_score queue: {precision_at_k:.1%}")
print("(Expected to be ~100% today — the queue is scored directly from this same definition.")
print(" That circularity is real, and it's why this isn't the final validation. See below.)")

# The meaningful comparison today: does traffic-awareness beat a rule with none at all,
# on the SAME yardstick both queues are being judged by?
naive_top_k = df.sort_values("content_age_days", ascending=False).head(K)
naive_hit = (
    (naive_top_k["trend_direction"] == "down")
    & (naive_top_k["impressions_90d"] >= 100)
    & (naive_top_k["sessions_90d"] > 0)
).mean()
print(f"\nPrecision@{K} of 'oldest content first' rule: {naive_hit:.1%}")
print("-> The gap between these two numbers is the honest signal today: traffic-aware")
print("   scoring finds real opportunities that a blind age-based rule mostly misses.")

Precision@100 of the priority_score queue: 100.0%
(Expected to be ~100% today — the queue is scored directly from this same definition.
 That circularity is real, and it's why this isn't the final validation. See below.)

Precision@100 of 'oldest content first' rule: 68.0%
-> The gap between these two numbers is the honest signal today: traffic-aware
   scoring finds real opportunities that a blind age-based rule mostly misses.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content page** (`content_id`), pseudonymized, from one of 32 clients,
with 90-day trailing search + engagement metrics. That's the grain the whole lane operates at:
the priority queue is a ranking *over pages*, so every feature, every score, and eventually every
model prediction has to live at this same one-page-per-row grain.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1] + 3} columns "
      f"(44 original + 3 I derived above)")
print(f"content_id is unique per row: {df['content_id'].is_unique}")
print(f"Distinct clients represented: {df['client_id'].nunique()}")

# The unit of analysis, as an actual dataframe
cols = [
    "content_id", "client_id", "content_type", "content_age_days",
    "impressions_90d", "clicks_90d", "sessions_90d", "trend_direction",
    "is_declining", "measurable_opportunity", "priority_score",
]
df[cols].sort_values("priority_score", ascending=False).head(10)

Shape: 30,000 rows x 50 columns (44 original + 3 I derived above)
content_id is unique per row: True
Distinct clients represented: 32


,content_id,client_id,content_type,content_age_days,impressions_90d,clicks_90d,sessions_90d,trend_direction,is_declining,measurable_opportunity,priority_score
6653,content_5fe46e04994d,client_4e07408562,keyword article,537,517715,741,520,down,1,1,13.158612
26844,content_8c19996aa890,client_4e07408562,keyword article,445,509252,785,571,down,1,1,13.142241
21819,content_4c36c775b818,client_4e07408562,keyword article,445,463103,1889,1114,down,1,1,13.049778
29879,content_1a9e894be2e2,client_19581e27de,keyword article,482,416180,944,1140,down,1,1,12.941141
13537,content_2c2606c5d176,client_19581e27de,keyword article,362,347399,1854,2146,down,1,1,12.763555
21565,content_9532f197bbc8,client_4e07408562,keyword article,445,309192,2689,1098,down,1,1,12.650380
26531,content_cb112fce36be,client_19581e27de,keyword article,126,309910,492,480,down,1,1,12.645627
27478,content_008fb02c46cb,client_349c41201b,keyword article,111,236803,605,703,down,1,1,12.377540
23767,content_813e88069237,client_6208ef0f77,keyword article,153,233561,129,1538,down,1,1,12.361755
26304,content_ff94c9b6b411,client_349c41201b,keyword article,154,228566,89,87,down,1,1,12.339974


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule has to pick a small number of thresholds up front and hope they generalize. This
lane has at least half a dozen signals that interact — `trend_pct`, `impressions_90d`,
`avg_position`, `engagement_rate`, `ai_traffic_pct`, `search_volume`, `content_type` — and they
don't move together. A page can be declining hard on impressions while its `ai_traffic_pct`
climbs (AI answer engines are starting to cite it), which is a completely different story than a
page declining everywhere at once. A rule like "sort by content_age_days" or even "declining +
impressions above X" can't represent that a *shallow* decline on a *huge*-traffic page might
outrank a *steep* decline on a tiny page, while a third page with the same traffic profile but
recovering AI referrals shouldn't be prioritized at all. Hand-tuning weights for every
interaction like that, and re-tuning them as client mixes and seasons shift, is exactly the
"real but too messy to write by hand" case the framing skill describes — that's where a model
earns its place over an if-statement.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Concrete case: two pages a simple "old + declining" rule would rank identically,
# but that clearly don't deserve the same review priority.
example_cols = [
    "content_id", "content_age_days", "trend_direction", "trend_pct",
    "impressions_90d", "ai_traffic_pct", "priority_score",
]

old_declining = df[(df["trend_direction"] == "down") & (df["content_age_days"] > 300)]
sample = (
    old_declining
    .sort_values("impressions_90d", ascending=False)
    [example_cols]
    .head(1)
)
sample2 = (
    old_declining
    .sort_values("impressions_90d", ascending=True)
    [example_cols]
    .head(1)
)
pd.concat([sample, sample2])

,content_id,content_age_days,trend_direction,trend_pct,impressions_90d,ai_traffic_pct,priority_score
6653,content_5fe46e04994d,537,down,-44.8,517715,0.38,13.158612
21952,content_fb40eae3b3a4,334,down,-100.0,1,0.00,0.000000


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.